# 06 · 随机数与可复现性 Random Numbers & Reproducibility

> **能力标签**：SH3（NumPy 与向量化计算 / NumPy & vectorized computing）

## 目标 Objectives

完成本课后，你应该能够：

1. 使用 `np.random.default_rng(seed)` 创建**可复现**的随机数生成器（Generator）。
2. 理解种子（seed）的作用，知道旧 API（`np.random.seed`）的缺陷。
3. 生成常见分布的随机样本：正态、均匀、整数。
4. 理解数值稳定性（numerical stability）的基本概念，并在 softmax 中应用。
5. 正确保存/恢复 RNG 状态（`rng.bit_generator.state`）。

> 对应能力：**SH3**


## 概念讲解 Concepts

### 随机数生成器 RNG

NumPy ≥ 1.17 推荐使用新 API：

```python
rng = np.random.default_rng(seed=42)   # PCG64 算法，现代化
```

旧 API `np.random.seed(42)` 修改**全局状态**，在多线程/多进程中不安全，已不推荐。

---

### 常见分布

```python
rng = np.random.default_rng(0)

rng.normal(loc=0.0, scale=1.0, size=(3, 4))   # 正态分布
rng.uniform(low=0.0, high=1.0, size=10)        # 均匀分布 [0,1)
rng.integers(0, 10, size=5)                    # 离散均匀整数 [0,10)
rng.choice(np.arange(20), size=5, replace=False)  # 无放回抽样
```

---

### 种子与可复现性 Seed & Reproducibility

```python
rng1 = np.random.default_rng(42)
rng2 = np.random.default_rng(42)
print(rng1.normal(size=3))   # 与 rng2 相同
print(rng2.normal(size=3))   # 完全一致
```

科研/工程中的最佳实践：
- 在实验入口处设定一个全局种子，存入日志。
- 不要在循环中重置种子（会导致伪随机）。
- 测试时固定种子，生产时可省略。

---

### 数值稳定性与 Softmax

Softmax 的数学定义：

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

当 $x_i$ 很大时，$e^{x_i}$ 溢出（overflow）为 `inf`。
**稳定技巧**：先减去最大值（不影响结果）：

$$\text{softmax}(x_i) = \frac{e^{x_i - \max x}}{\sum_j e^{x_j - \max x}}$$

---

### RNG 状态保存

```python
state = rng.bit_generator.state   # 保存快照
rng.normal(size=5)                 # 消耗随机数
rng.bit_generator.state = state   # 恢复
rng.normal(size=5)                 # 与之前完全相同
```


## 示例 Worked Example 1：可复现随机实验

In [ ]:
import numpy as np

# ── 旧 API vs 新 API ─────────────────────────────────────────────────────
# 旧（不推荐，全局状态）
# np.random.seed(42)
# arr_old = np.random.normal(size=5)

# 新（推荐）
rng = np.random.default_rng(42)
arr = rng.normal(size=5)
print("正态样本（seed=42）:", arr.round(4))

# 用相同 seed 重复 → 完全一致
rng2 = np.random.default_rng(42)
arr2 = rng2.normal(size=5)
print("重复实验一致:", np.allclose(arr, arr2))

# 不同分布
rng3 = np.random.default_rng(0)
print("\n均匀 [0,1):", rng3.uniform(size=4).round(4))
print("整数 [0,10):", rng3.integers(0, 10, size=6))
print("无放回抽样:", rng3.choice(20, size=5, replace=False))


## 示例 Worked Example 2：数值稳定的 Softmax

In [ ]:
import numpy as np

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """数值稳定的 softmax（先减去每行最大值）。"""
    x = np.asarray(x, dtype=float)
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

# ── 演示数值稳定性 ────────────────────────────────────────────────────────
x_large = np.array([1000.0, 1001.0, 1002.0])

# 不稳定版本
e_naive = np.exp(x_large)
print("不稳定版 exp(x):", e_naive)      # [inf, inf, inf]

# 稳定版本
s = softmax(x_large)
print("稳定版 softmax:", s.round(4))    # 正常结果
print("和为 1:", np.allclose(s.sum(), 1.0))

# ── 批量（矩阵）softmax ───────────────────────────────────────────────────
rng = np.random.default_rng(7)
X = rng.normal(size=(4, 5))
S = softmax(X, axis=-1)    # 对每行做 softmax
print("\n批量 softmax shape:", S.shape)
print("每行之和:", S.sum(axis=1).round(10))
print("所有值正数:", (S > 0).all())


In [ ]:
import numpy as np

# ── RNG 状态保存与恢复 ───────────────────────────────────────────────────
rng = np.random.default_rng(100)

state = rng.bit_generator.state          # 保存快照
first_draw = rng.normal(size=5)
print("第一次抽样:", first_draw.round(4))

rng.bit_generator.state = state          # 恢复到快照
second_draw = rng.normal(size=5)
print("恢复后抽样:", second_draw.round(4))
print("两次结果一致:", np.allclose(first_draw, second_draw))

# ── 蒙特卡洛估计 π ──────────────────────────────────────────────────────
rng_mc = np.random.default_rng(2024)
N = 1_000_000
pts = rng_mc.uniform(-1, 1, size=(N, 2))
inside = (pts[:, 0]**2 + pts[:, 1]**2) <= 1.0
pi_est = 4.0 * inside.sum() / N
print(f"\n蒙特卡洛估计 π ≈ {pi_est:.5f}  (真实值: {np.pi:.5f})")


## 动手 Exercise

用 `np.random.default_rng` 模拟**Bootstrap 置信区间**：

1. 生成 `n=200` 个正态分布（均值 5，标准差 2）的样本。
2. 进行 `B=1000` 次有放回抽样（每次 `n` 个），计算每次的样本均值。
3. 用 `np.percentile` 计算 95% Bootstrap 置信区间。


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 200
B = 1000

# 1. 生成原始样本
data = rng.normal(loc=5.0, scale=2.0, size=n)
print(f"样本均值: {data.mean():.4f}  (真实: 5.0)")

# 2. Bootstrap 重采样（向量化）
# shape (B, n) — 每行是一次有放回抽样的索引
boot_idx = rng.integers(0, n, size=(B, n))
boot_means = data[boot_idx].mean(axis=1)  # (B,) — 每次 bootstrap 均值

# 3. 95% 置信区间
ci_low  = np.percentile(boot_means, 2.5)
ci_high = np.percentile(boot_means, 97.5)
print(f"\nBootstrap 95% CI: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"区间宽度: {ci_high - ci_low:.4f}")


## 延伸阅读 Further Reading

1. **NumPy 新 RNG 文档**: <https://numpy.org/doc/stable/reference/random/generator.html>
2. **PCG64 算法简介**: <https://www.pcg-random.org/>
3. **数值稳定性 — "The Art of Doing Science and Engineering"** — R. Hamming
4. **Bootstrap 方法**: Efron & Hastie 《Computer Age Statistical Inference》第 10 章（免费在线）
5. **softmax 的数值问题**: CS231n Lecture Notes — Linear Classification


## 关联练习 Related Assignments

本课对应练习包：**`w2-softmax`**（任务：数值稳定的 `softmax`）

```bash
python check.py w2-softmax
```

> 能力标签：**SH3** · threshold ≥ 0.7
